# LeetCode 877 — Stone Game bằng Python

Link đề: https://leetcode.com/problems/stone-game/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Game Theory, Minimax, Zero-Sum Game

Skeleton LeetCode:

```python
class Solution:
    def stoneGame(self, piles: List[int]) -> bool:
        ...
```

Ghi chú: đáp án là `bool`. Điều thú vị: với mọi đầu vào hợp lệ (số đống **chẵn**, tổng **lẻ**) đáp án luôn là `True` (Alice thắng). Notebook vừa giải bằng quy hoạch động theo đoạn (cách tổng quát, tính được **hiệu số điểm** tối ưu), vừa chứng minh và kiểm chứng bằng máy công thức `return True`. Oracle: **minimax vét cạn** trên cây trò chơi, mỗi người chơi tối đa hoá số đá của chính mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai người lần lượt lấy **cả một đống** ở **đầu hoặc cuối** hàng; ai có nhiều đá hơn thắng; cả hai chơi tối ưu; Alice đi trước.
2. Nắm công cụ Python: đệ quy với bộ nhớ `functools.lru_cache`, quy hoạch động theo **đoạn** `[i, j]`, mảng một chiều thay bảng hai chiều, `itertools.product` để vét cạn.
3. Đi từ **minimax vét cạn (`O(2^n)`)** → **DP có nhớ theo đoạn** (hiệu số điểm) → **DP bảng từ dưới lên** → **`return True`** (chứng minh bằng chẵn/lẻ chỉ số).
4. Hiểu mấu chốt: trò chơi **tổng bằng không**, nên đặt `f(i, j)` = (điểm người đi **hiện tại** − điểm người kia) trên đoạn `[i, j]` khi cả hai chơi tối ưu: `f(i, j) = max(p[i] − f(i+1, j), p[j] − f(i, j−1))`.
5. Tránh lỗi kinh điển: tham lam (lấy đống lớn hơn ở hai đầu) sai, cộng thay vì **trừ** `f` của đối thủ, chỉ xét một "tính chẵn lẻ" cố định, nhầm kết luận `True` khi số đống **lẻ** hoặc tổng **chẵn**, và lấy đống ở giữa.

## 2. Đề bài & ví dụ

Alice và Bob chơi với một hàng các đống đá. Số đống là **chẵn**, mỗi đống có số đá **nguyên dương**. Tổng số đá là **lẻ** nên không hoà.

Hai người luân phiên, **Alice đi trước**. Mỗi lượt lấy **cả đống** ở **đầu** hoặc **cuối** hàng. Khi hết đống, ai có nhiều đá hơn thắng.

Cả hai chơi **tối ưu**. Trả về `true` nếu Alice thắng, `false` nếu Bob thắng.

**Ví dụ 1:** `piles = [5,3,4,5]` → `true`.

**Ví dụ 2:** `piles = [3,7,2,3]` → `true`.

**Ràng buộc:** `2 <= piles.length <= 500`; `piles.length` chẵn; `1 <= piles[i] <= 500`; `sum(piles)` lẻ.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [([5, 3, 4, 5], True), ([3, 7, 2, 3], True)]
for p, want in examples:
    print(f"piles = {p} -> {want}")

piles = [5, 3, 4, 5] -> True
piles = [3, 7, 2, 3] -> True


## 3. Kiến thức nền

### 3.1 Trò chơi tổng bằng không và minimax

Tổng số đá cố định `S`: nếu Alice có `a` thì Bob có `S − a`. Alice muốn **tăng** `a − (S − a)`, Bob muốn **giảm** nó. Đây là trò chơi **tổng bằng không**: mỗi người tối đa hoá điểm của mình ⇔ tối đa hoá **hiệu số** của mình trừ đối thủ. Phương pháp: **minimax** (mỗi nước đi giả định đối thủ đáp lại tối ưu).

### 3.2 Biểu diễn trạng thái bằng đoạn `[i, j]`

Sau mỗi lượt, các đống còn lại luôn là một **đoạn liên tiếp** `piles[i..j]`. Trạng thái = `(i, j)` (và ai đang đi, nhưng ta gộp nhờ cách định nghĩa hiệu số). Có `O(n²)` trạng thái.

In [2]:
def remaining_after(piles, moves):
    """moves: chuỗi 'L'/'R'; trả đoạn còn lại sau các nước đó."""
    i, j = 0, len(piles) - 1
    for m in moves:
        if m == "L":
            i += 1
        else:
            j -= 1
    return piles[i:j + 1]


p = [5, 3, 4, 5]
for mv in ["", "L", "R", "LR", "RL", "LL", "RR"]:
    print(f"nước {mv!r:5s} -> còn {remaining_after(p, mv)}")

nước ''    -> còn [5, 3, 4, 5]
nước 'L'   -> còn [3, 4, 5]
nước 'R'   -> còn [5, 3, 4]
nước 'LR'  -> còn [3, 4]
nước 'RL'  -> còn [3, 4]
nước 'LL'  -> còn [4, 5]
nước 'RR'  -> còn [5, 3]


### 3.3 Hiệu số tương đối `f(i, j)`

`f(i, j)` = (số đá người **đi hiện tại** nhận được) − (số đá đối thủ nhận được) trên đoạn `[i, j]` khi cả hai tối ưu. Người đi chọn:

- lấy `piles[i]`, rồi đối thủ đi trên `[i+1, j]` với hiệu số `f(i+1, j)` (từ góc nhìn đối thủ) ⇒ hiệu số của ta: `piles[i] − f(i+1, j)`;
- hoặc lấy `piles[j]`: `piles[j] − f(i, j−1)`.

`f(i, j) = max(piles[i] − f(i+1, j), piles[j] − f(i, j−1))`, `f(i, i) = piles[i]`. Alice thắng ⇔ `f(0, n−1) > 0`.

### 3.4 `lru_cache`

Decorator `@lru_cache(None)` lưu kết quả theo tham số, biến đệ quy lặp thành DP.

In [3]:
@lru_cache(None)
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)


print(fib_cached(60), fib_cached.cache_info().hits, "lần dùng lại kết quả")

1548008755920 58 lần dùng lại kết quả


## 4. Ý tưởng nền tảng / trực giác

`piles = [5, 3, 4, 5]`. Alice lấy `5` (đầu). Còn `[3, 4, 5]`:

- Bob lấy `3` → còn `[4, 5]`; Alice lấy `5`: Alice `10`, Bob `3 + 4 = 7`.
- Bob lấy `5` → còn `[3, 4]`; Alice lấy `4`: Alice `9`, Bob `8`.

Bob tối ưu chọn nước làm Alice được ít nhất: `9` vs `8`. Alice vẫn thắng.

Sẽ thấy có một **quy luật chẵn/lẻ**: chỉ số `0, 2` (vị trí chẵn): `5 + 4 = 9`; chỉ số `1, 3` (vị trí lẻ): `3 + 5 = 8`. Alice có thể chọn sao cho **luôn** lấy đúng các đống chẵn (hoặc đúng các đống lẻ). Vì tổng lẻ nên một trong hai nhóm lớn hơn nhóm kia — Alice chọn nhóm lớn hơn và thắng. Đó là lý do đáp án luôn `True`.

In [4]:
p = [5, 3, 4, 5]
even, odd = sum(p[0::2]), sum(p[1::2])
print("tổng chỉ số chẵn:", even, "| tổng chỉ số lẻ:", odd, "| tổng:", even + odd, "(lẻ nên even != odd)")

tổng chỉ số chẵn: 9 | tổng chỉ số lẻ: 8 | tổng: 17 (lẻ nên even != odd)


## 5. Cách 1 — Brute Force: minimax vét cạn

**Ý tưởng:** đệ quy không nhớ. `solve(i, j)` trả `(số đá của người đi, số đá của người kia)` từ các đống `[i, j]`. Người đi thử cả hai nước, chọn nước cho **chính họ** nhiều đá hơn (đối thủ trả lời tối ưu theo cùng quy tắc).

In [5]:
def stone_minimax(piles):
    def solve(i, j):
        if i > j:
            return 0, 0
        # lấy đầu: mình +piles[i]; sau đó người kia là "người đi" trên [i+1, j]
        o1, m1 = solve(i + 1, j)
        take_left = (piles[i] + m1, o1)
        # lấy cuối
        o2, m2 = solve(i, j - 1)
        take_right = (piles[j] + m2, o2)
        return max(take_left, take_right)                 # mình chọn nước cho MÌNH nhiều hơn

    alice, bob = solve(0, len(piles) - 1)
    return alice > bob


for p, want in examples:
    assert stone_minimax(p) == want
print([stone_minimax(p) for p, _ in examples])

[True, True]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(2^n): chỉ dùng cho n nhỏ)
class SolutionMinimax:
    def stoneGame(self, piles: List[int]) -> bool:
        def solve(i, j):
            if i > j:
                return 0, 0
            o1, m1 = solve(i + 1, j)
            o2, m2 = solve(i, j - 1)
            return max((piles[i] + m1, o1), (piles[j] + m2, o2))

        a, b = solve(0, len(piles) - 1)
        return a > b

**Độ phức tạp:** mỗi lời gọi sinh hai lời gọi nhỏ hơn một đống: `O(2^n)` thời gian, `O(n)` ngăn xếp. Với `n = 500` là không thể.

**Nút thắt:** cùng đoạn `[i, j]` được tính lặp lại rất nhiều lần (hoán vị thứ tự lấy hai đầu cho cùng đoạn).

## 6. Ý tưởng thuật toán / bước đột phá

Hai hướng:

1. **DP theo đoạn:** chỉ có `O(n²)` đoạn `[i, j]`; tính `f(i, j) = max(p[i] − f(i+1, j), p[j] − f(i, j−1))` một lần mỗi đoạn. Đáp án `f(0, n−1) > 0`.
2. **Toán:** Alice có chiến lược ép lấy toàn bộ các đống chỉ số chẵn hoặc toàn bộ chỉ số lẻ; tổng lẻ ⇒ một nhóm hơn nhóm kia ⇒ Alice luôn thắng ⇒ `return True`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** DP có nhớ (top-down), **Cách 3** DP bảng một chiều (bottom-up), **Cách 4** `return True` kèm chứng minh.

### Cách 2 — DP có nhớ (top-down) trên hiệu số

**Giải thích từng đoạn:**

1. `f(i, j)` = hiệu số (người đi − người kia) trên `[i, j]`.
2. `f(i, i) = p[i]`.
3. `f(i, j) = max(p[i] − f(i+1, j), p[j] − f(i, j−1))`.
4. Alice thắng ⇔ `f(0, n−1) > 0`.

In [7]:
def stone_dp_memo(piles):
    @lru_cache(None)
    def f(i, j):
        if i == j:
            return piles[i]
        return max(piles[i] - f(i + 1, j), piles[j] - f(i, j - 1))

    return f(0, len(piles) - 1) > 0


for p, want in examples:
    assert stone_dp_memo(p) == want
print([stone_dp_memo(p) for p, _ in examples])

[True, True]


In [8]:
# Code nộp LeetCode cho Cách 2
from functools import lru_cache


class SolutionMemo:
    def stoneGame(self, piles: List[int]) -> bool:
        @lru_cache(None)
        def f(i, j):
            if i == j:
                return piles[i]
            return max(piles[i] - f(i + 1, j), piles[j] - f(i, j - 1))

        return f(0, len(piles) - 1) > 0

**Độ phức tạp:** `O(n²)` thời gian và bộ nhớ; ngăn xếp đệ quy sâu tới `n = 500` (vượt giới hạn mặc định `1000`? độ sâu 500 vẫn ổn).

### Cách 3 — DP bảng một chiều (bottom-up)

**Ý tưởng:** duyệt độ dài đoạn tăng dần. `dp[i]` giữ `f(i, i + len − 1)` của độ dài hiện tại; cập nhật `dp[i] = max(p[i] − dp[i+1], p[i+len−1] − dp[i])` (với `dp[i+1]` là `f(i+1, j)` của độ dài trước, `dp[i]` là `f(i, j−1)` cũ). Duyệt `i` tăng để không ghi đè giá trị còn cần.

In [9]:
def stone_dp_table(piles):
    n = len(piles)
    dp = piles[:]                                         # độ dài 1: f(i, i) = piles[i]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            # dp[i] hiện là f(i, j-1); dp[i+1] là f(i+1, j)
            dp[i] = max(piles[i] - dp[i + 1], piles[j] - dp[i])
    return dp[0] > 0


for p, want in examples:
    assert stone_dp_table(p) == want
print([stone_dp_table(p) for p, _ in examples])

[True, True]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionTable:
    def stoneGame(self, piles: List[int]) -> bool:
        n = len(piles)
        dp = piles[:]
        for length in range(2, n + 1):
            for i in range(n - length + 1):
                j = i + length - 1
                dp[i] = max(piles[i] - dp[i + 1], piles[j] - dp[i])
        return dp[0] > 0

**Độ phức tạp:** `O(n²)` thời gian, `O(n)` bộ nhớ, không đệ quy.

### Cách 4 — `return True` (chứng minh bằng chẵn/lẻ)

**Ý tưởng:** số đống `n` chẵn. Đánh chỉ số `0..n−1`. Ở **lượt của Alice** số đống còn lại chẵn nên hai đầu đoạn `i, j` có `j − i` lẻ, tức chỉ số của hai đầu **khác tính chẵn lẻ**. Alice luôn có thể lấy đầu có chỉ số **chẵn**. Sau đó còn lẻ đống; hai đầu cùng tính chẵn lẻ (đều lẻ), Bob buộc lấy đống chỉ số **lẻ**. Lặp lại: Alice thu **toàn bộ** chỉ số chẵn, Bob toàn bộ chỉ số lẻ. Alice cũng có thể làm ngược lại (toàn bộ chỉ số lẻ). Vì `Σ chẵn + Σ lẻ = S` lẻ nên chúng khác nhau; Alice chọn nhóm lớn hơn ⇒ thắng.

In [11]:
def stone_true(piles):
    return True


for p, want in examples:
    assert stone_true(p) == want
print([stone_true(p) for p, _ in examples])

[True, True]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionTrue:
    def stoneGame(self, piles: List[int]) -> bool:
        return True

**Độ phức tạp:** `O(1)` thời gian và bộ nhớ. Đúng **chỉ khi** đầu vào thoả ràng buộc (số đống chẵn, tổng lẻ).

Ta còn kiểm chứng **chiến lược chẵn/lẻ** bằng máy ở mục 10 và 12.

## 8. TRACE từng bước

**Trace Cách 3 (bảng)** với `piles = [5, 3, 4, 5]`:

- `dp = [5, 3, 4, 5]` (độ dài 1).
- Độ dài 2: `dp[0] = max(5 − 3, 3 − 5) = 2`; `dp[1] = max(3 − 4, 4 − 3) = 1`; `dp[2] = max(4 − 5, 5 − 4) = 1` ⇒ `dp = [2, 1, 1, 5]` (3 phần tử đầu là `f` độ dài 2).
- Độ dài 3: `dp[0] = max(5 − dp[1], 4 − dp[0]) = max(4, 2) = 4` (`j = 2`); `dp[1] = max(3 − dp[2], 5 − dp[1]) = max(2, 4) = 4` (`j = 3`) ⇒ `[4, 4, ...]`.
- Độ dài 4: `dp[0] = max(5 − dp[1], 5 − dp[0]) = max(1, 1) = 1` ⇒ `f(0, 3) = 1 > 0`.

Hiệu số `1`: Alice `9`, Bob `8` ✓ (khớp ví dụ). Hàm trace in từng bước:

In [13]:
def trace_dp(piles):
    n = len(piles)
    dp = piles[:]
    print("độ dài 1:", dp)
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            a, b = piles[i] - dp[i + 1], piles[j] - dp[i]
            print(f"  f({i},{j}) = max(p[{i}] - f({i + 1},{j}), p[{j}] - f({i},{j - 1})) = max({piles[i]} - {dp[i + 1]}, {piles[j]} - {dp[i]}) = max({a}, {b})")
            dp[i] = max(a, b)
        print(f"độ dài {length}: dp[:{n - length + 1}] =", dp[:n - length + 1])
    print("=> f(0, n-1) =", dp[0], "-> Alice", "thắng" if dp[0] > 0 else "thua", "(Alice - Bob =", dp[0], ")")
    return dp[0] > 0


assert trace_dp([5, 3, 4, 5]) is True

độ dài 1: [5, 3, 4, 5]
  f(0,1) = max(p[0] - f(1,1), p[1] - f(0,0)) = max(5 - 3, 3 - 5) = max(2, -2)
  f(1,2) = max(p[1] - f(2,2), p[2] - f(1,1)) = max(3 - 4, 4 - 3) = max(-1, 1)
  f(2,3) = max(p[2] - f(3,3), p[3] - f(2,2)) = max(4 - 5, 5 - 4) = max(-1, 1)
độ dài 2: dp[:3] = [2, 1, 1]
  f(0,2) = max(p[0] - f(1,2), p[2] - f(0,1)) = max(5 - 1, 4 - 2) = max(4, 2)
  f(1,3) = max(p[1] - f(2,3), p[3] - f(1,2)) = max(3 - 1, 5 - 1) = max(2, 4)
độ dài 3: dp[:2] = [4, 4]
  f(0,3) = max(p[0] - f(1,3), p[3] - f(0,2)) = max(5 - 4, 5 - 4) = max(1, 1)
độ dài 4: dp[:1] = [1]
=> f(0, n-1) = 1 -> Alice thắng (Alice - Bob = 1 )


**Trace chiến lược chẵn/lẻ** (Cách 4): với `[5, 3, 4, 5]` nhóm chẵn `{5, 4} = 9`, nhóm lẻ `{3, 5} = 8` ⇒ Alice chọn chỉ số chẵn: lượt 1 lấy chỉ số `0` (chẵn), Bob buộc lấy chỉ số `1` hoặc `3` (đều lẻ); nếu Bob lấy chỉ số `1`, còn `[2, 3]`: Alice lấy chỉ số `2` (chẵn); Bob lấy `3`. Alice `5 + 4 = 9` ✓.

In [14]:
def trace_parity(piles, parity):
    """Alice theo chiến lược lấy đống có chỉ số gốc cùng tính chẵn lẻ `parity`; Bob lấy đầu bên trái (ví dụ)."""
    i, j = 0, len(piles) - 1
    alice = bob = 0
    turn = "Alice"
    while i <= j:
        if turn == "Alice":
            idx = i if i % 2 == parity else j                 # đúng MỘT đầu có tính chẵn lẻ mong muốn
            assert idx % 2 == parity
            alice += piles[idx]
        else:
            idx = i                                           # Bob: lấy đầu trái (bất kỳ nước nào cũng thuộc nhóm còn lại)
            assert idx % 2 != parity or idx == j
            bob += piles[idx]
        print(f"{turn:5s} lấy chỉ số {idx} (đá = {piles[idx]}) | còn [{i}..{j}]")
        if idx == i:
            i += 1
        else:
            j -= 1
        turn = "Bob" if turn == "Alice" else "Alice"
    print("Alice =", alice, ", Bob =", bob)
    return alice, bob


a, b = trace_parity([5, 3, 4, 5], 0)
assert (a, b) == (9, 8)

Alice lấy chỉ số 0 (đá = 5) | còn [0..3]
Bob   lấy chỉ số 1 (đá = 3) | còn [1..3]
Alice lấy chỉ số 2 (đá = 4) | còn [2..3]
Bob   lấy chỉ số 3 (đá = 5) | còn [3..3]
Alice = 9 , Bob = 8


## 9. Code nộp LeetCode

Chọn **Cách 4** (đúng theo ràng buộc, `O(1)`). Ta cũng giữ bản DP làm công cụ tổng quát ở các mục sau. Giữ nguyên `class Solution`, `stoneGame(self, piles)`:

In [15]:
class Solution:
    def stoneGame(self, piles: List[int]) -> bool:
        # n chẵn, tổng lẻ: Alice có thể ép lấy toàn bộ chỉ số chẵn hoặc toàn bộ chỉ số lẻ,
        # hai nhóm có tổng khác nhau (tổng lẻ) nên nhóm lớn hơn thuộc về Alice.
        return True


print([Solution().stoneGame(p) for p, _ in examples])                     # [True, True]
assert all(Solution().stoneGame(p) == want for p, want in examples)

[True, True]


## 10. Vì sao đúng

**(a) Hiệu số tối ưu.** Gọi `f(i, j)` như mục 3.3. Quy nạp theo độ dài: độ dài 1 người đi lấy đống duy nhất: `f(i, i) = p[i]`. Độ dài `L > 1`: người đi lấy `p[i]` rồi đối thủ có hiệu số tối ưu `f(i+1, j)` trên phần còn lại, nên hiệu số của người đi là `p[i] − f(i+1, j)`; tương tự lấy `p[j]`. Người đi chọn nước tốt hơn. Vì tổng bằng không, tối đa hoá điểm mình ⇔ tối đa hoá hiệu số. Alice thắng ⇔ `f(0, n−1) > 0` (hiệu số `≠ 0` vì tổng lẻ).

**(b) Chiến lược chẵn/lẻ.** Xét lượt của Alice khi còn `2m` đống liên tiếp ở chỉ số `i..j`. Khi đó `j − i = 2m − 1` là số lẻ nên `i` và `j` **khác** tính chẵn lẻ: đúng một đầu có chỉ số chẵn. Alice lấy đầu chẵn đó. Còn lại `2m − 1` đống, hai đầu mới `i', j'` có `j' − i'` chẵn nên **cùng** tính chẵn lẻ; một trong hai là đầu lẻ mà Alice bỏ lại (chỉ số lẻ), vậy cả hai đầu đều lẻ và Bob **buộc** lấy một đống chỉ số lẻ. Quay lại tình huống `2(m−1)` đống ở lượt Alice. Quy nạp: Alice thu **mọi** đống chỉ số chẵn, Bob thu mọi đống chỉ số lẻ. Đối xứng, nếu Alice luôn lấy đầu có chỉ số lẻ thì cô thu mọi đống lẻ (nước đầu tiên lấy chỉ số `n − 1`, lẻ).

**(c) Kết luận.** Tổng `S = E + O` (chẵn + lẻ chỉ số) là lẻ ⇒ `E ≠ O`. Alice chọn nhóm lớn hơn và **bất kể Bob chơi thế nào** thu đúng nhóm đó ⇒ thắng. Vậy đáp án luôn `True` khi `n` chẵn và `S` lẻ.

**(d) Khi ràng buộc bị phá vỡ.** Với `n` **lẻ**, Alice có thể thua (ví dụ ở mục 15: có phản ví dụ nhỏ); với tổng **chẵn**, Alice chỉ đảm bảo **không thua** (`f ≥ 0`). Cách 4 phụ thuộc hoàn toàn vào ràng buộc, còn DP (Cách 2/3) đúng mọi trường hợp.

Ta kiểm chứng bằng máy: **mọi** `piles` nhỏ thoả ràng buộc cho `True` khớp minimax vét cạn (mục 12D), chiến lược chẵn/lẻ thắng với **mọi** cách đi của Bob (12E), và DP khớp minimax trên cả đầu vào *không* thoả ràng buộc.

In [16]:
def parity_strategy_wins_all_bob_moves(piles):
    """Vét cạn MỌI cách đi của Bob: Alice theo chiến lược chọn nhóm chẵn/lẻ lớn hơn luôn thu đúng nhóm đó và thắng."""
    n = len(piles)
    E, O = sum(piles[0::2]), sum(piles[1::2])
    parity = 0 if E > O else 1

    def play(i, j, alice, turn):
        if i > j:
            return alice * 2 > sum(piles)
        if turn == 0:                                          # Alice: lấy đầu có chỉ số cùng tính chẵn lẻ với `parity`
            if i % 2 == parity:
                return play(i + 1, j, alice + piles[i], 1)
            assert j % 2 == parity
            return play(i, j - 1, alice + piles[j], 1)
        return play(i + 1, j, alice, 0) and play(i, j - 1, alice, 0)    # Bob thử cả hai nước

    return play(0, n - 1, 0, 0)


print(parity_strategy_wins_all_bob_moves([5, 3, 4, 5]), parity_strategy_wins_all_bob_moves([3, 7, 2, 3]))

True True


## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Minimax vét cạn | `O(2^n)` | `O(n)` | oracle, `n ≤ ~20` |
| 2. DP có nhớ | `O(n²)` | `O(n²)` | tổng quát |
| 3. DP bảng một chiều | `O(n²)` | `O(n)` | tổng quát, tiết kiệm |
| 4. `return True` | `O(1)` | `O(1)` | **đáp án chính**, chỉ khi thoả ràng buộc |

Đo thực tế (`n = 500`; minimax chỉ `n = 18`):

In [17]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


def random_valid(rng, n):
    while True:
        p = [rng.randint(1, 500) for _ in range(n)]
        if sum(p) % 2 == 1:
            return p


rng = random.Random(877)
big = random_valid(rng, 500)
print(f"n = 500: memo {bench(stone_dp_memo, big, 1) * 1e3:7.2f} ms | table {bench(stone_dp_table, big) * 1e3:7.2f} ms | true {bench(stone_true, big) * 1e6:5.2f} us")
mid = random_valid(rng, 18)
print(f"n = 18 : minimax {bench(stone_minimax, mid, 1) * 1e3:8.1f} ms (2^18 nhánh)")

n = 500: memo  115.76 ms | table   29.68 ms | true  0.14 us
n = 18 : minimax     94.3 ms (2^18 nhánh)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi đầu vào nhỏ hợp lệ với oracle minimax, (E) ngẫu nhiên có seed + kiểm chứng chiến lược chẵn/lẻ, (F) giới hạn `n = 500`.

### A. Ví dụ của đề

In [18]:
def oracle(piles):
    """Minimax đúng đề: mỗi người tối đa hoá số đá CỦA MÌNH (không dùng hiệu số). Trả (alice_total, bob_total)."""
    def solve(i, j):                                    # (người đi hiện tại, người kia) trên đoạn [i, j]
        if i > j:
            return 0, 0
        o1, m1 = solve(i + 1, j)
        o2, m2 = solve(i, j - 1)
        return max((piles[i] + m1, o1), (piles[j] + m2, o2))

    return solve(0, len(piles) - 1)


IMPLS = {
    "minimax": stone_minimax,
    "memo": stone_dp_memo,
    "table": stone_dp_table,
    "true": stone_true,
    "Solution": lambda p: Solution().stoneGame(list(p)),
    "SolutionMinimax": lambda p: SolutionMinimax().stoneGame(list(p)),
    "SolutionMemo": lambda p: SolutionMemo().stoneGame(list(p)),
    "SolutionTable": lambda p: SolutionTable().stoneGame(list(p)),
    "SolutionTrue": lambda p: SolutionTrue().stoneGame(list(p)),
}


def check(piles, names=None):
    a, b = oracle(piles)
    want = a > b
    for name in (names or IMPLS):
        assert IMPLS[name](piles) == want, (name, piles, a, b)


for p, want in examples:
    a, b = oracle(p)
    assert (a > b) == want
    check(p)
print("ví dụ đề: oracle minimax (Alice vs Bob):", [oracle(p) for p, _ in examples], "và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle minimax (Alice vs Bob): [(9, 8), (10, 5)] và 9 cài đặt đúng


### B. Edge case

- `n = 2`: Alice lấy đống lớn hơn.
- Mọi đống bằng nhau (nhưng tổng lẻ ⇒ không thể với `n` chẵn; dùng giá trị gần bằng nhau).
- Đống rất lệch (`[1, 500]`, `[500, 1, 1, 500]`... kiểm tổng lẻ).
- Đống tăng dần, giảm dần.
- Đống lớn ở giữa (Alice không lấy được ngay): `[1, 100, 100, 1, ...]`.
- Đống toàn `1` (tổng lẻ cần `n` lẻ ⇒ không hợp lệ): chỉ kiểm tra thuộc ràng buộc.

In [19]:
cases = [[1, 2], [2, 1], [1, 500], [500, 1, 1, 500, 1, 2], [1, 2, 3, 4, 5, 8],
         [1, 100, 100, 1, 1, 2], [3, 7, 2, 3], [5, 3, 4, 5], [2, 2, 2, 3], [1, 1, 1, 2]]
for p in cases:
    assert len(p) % 2 == 0 and sum(p) % 2 == 1, p
    check(p)
    assert oracle(p)[0] > oracle(p)[1]                      # Alice thắng thật theo minimax
print(len(cases), "edge case hợp lệ: Alice luôn thắng; mọi cài đặt đúng")

10 edge case hợp lệ: Alice luôn thắng; mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_greedy_both(piles):                            # cả hai tham lam: lấy đầu lớn hơn
    i, j = 0, len(piles) - 1
    a = b = 0
    alice_turn = True
    while i <= j:
        if piles[i] >= piles[j]:
            x, i = piles[i], i + 1
        else:
            x, j = piles[j], j - 1
        if alice_turn:
            a += x
        else:
            b += x
        alice_turn = not alice_turn
    return a > b


def bug_alice_greedy(piles):                           # Alice tham lam, Bob tối ưu
    def best_diff(i, j):                               # hiệu số (người đi - người kia) với cả hai tối ưu
        if i > j:
            return 0
        return max(piles[i] - best_diff(i + 1, j), piles[j] - best_diff(i, j - 1))

    i, j = 0, len(piles) - 1
    if piles[i] >= piles[j]:
        first, rest = piles[i], best_diff(i + 1, j)
    else:
        first, rest = piles[j], best_diff(i, j - 1)
    return first - rest > 0


def bug_fixed_even(piles):                             # luôn chọn nhóm chỉ số chẵn (không chọn nhóm lớn hơn)
    return sum(piles[0::2]) > sum(piles[1::2])


def bug_add_not_sub(piles):                            # f = max(p[i] + f(i+1,j), ...) -> cộng thay vì trừ
    @lru_cache(None)
    def f(i, j):
        if i > j:
            return 0
        return max(piles[i] + f(i + 1, j), piles[j] + f(i, j - 1))

    return f(0, len(piles) - 1) > 0


def bug_take_middle(piles):                            # cho phép lấy đống ở giữa (sai luật)
    @lru_cache(None)
    def f(rest):
        if not rest:
            return 0
        return max(rest[k] - f(rest[:k] + rest[k + 1:]) for k in range(len(rest)))

    return f(tuple(piles)) > 0


BUGS = [bug_greedy_both, bug_alice_greedy, bug_fixed_even]      # bug_add_not_sub xem mục 13: trên đầu vào hợp lệ nó "vô tình" đúng
rng = random.Random(3)
tests = []
while len(tests) < 3000:
    n = rng.choice([2, 4, 6])
    p = [rng.randint(1, 9) for _ in range(n)]
    if sum(p) % 2:
        tests.append(p)
for bug in BUGS:
    bad = next((p for p in tests if bug(p) is not True), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên piles = {bad}: ra {bug(bad)}, đúng True (oracle {oracle(bad)})")
print("bug_take_middle: n = 4 ví dụ [3,7,2,3] ->", bug_take_middle([3, 7, 2, 3]), "(luật sai nhưng có thể vẫn ra True; xem mục 13)")

bug_greedy_both    sai trên piles = [4, 5, 7, 5]: ra False, đúng True (oracle (11, 10))
bug_alice_greedy   sai trên piles = [4, 5, 7, 5]: ra False, đúng True (oracle (11, 10))
bug_fixed_even     sai trên piles = [8, 2, 1, 8]: ra False, đúng True (oracle (10, 9))
bug_take_middle: n = 4 ví dụ [3,7,2,3] -> True (luật sai nhưng có thể vẫn ra True; xem mục 13)


### D. Vét cạn mọi đầu vào nhỏ hợp lệ

Mọi `piles` hợp lệ (số đống chẵn, tổng lẻ): `n = 2` với giá trị `1..9`, `n = 4` với `1..6`, `n = 6` với `1..4`, `n = 8` với `1..3`. Oracle: minimax. Kết quả: Alice **luôn** thắng, và mọi cài đặt khớp.

In [21]:
t0 = time.perf_counter()
cnt = 0
fast = ("memo", "table", "true", "Solution", "SolutionMemo", "SolutionTable", "SolutionTrue")
for n, hi in [(2, 9), (4, 6), (6, 4), (8, 3)]:
    for p in itertools.product(range(1, hi + 1), repeat=n):
        if sum(p) % 2 == 0:
            continue
        p = list(p)
        a, b = oracle(p)
        assert a > b, p                                  # Alice thắng thật theo minimax
        for name in fast:
            assert IMPLS[name](p) is True, (name, p)
        assert stone_minimax(p) is True
        cnt += 1
print(cnt, "đầu vào hợp lệ: minimax cho Alice thắng ở TẤT CẢ; 8 cài đặt đều True;", round(time.perf_counter() - t0, 1), "s")

6016 đầu vào hợp lệ: minimax cho Alice thắng ở TẤT CẢ; 8 cài đặt đều True; 1.0 s


### E. Ngẫu nhiên có seed và chiến lược chẵn/lẻ

Đầu vào hợp lệ ngẫu nhiên cỡ tới `n = 14` so với minimax; thêm kiểm chứng **chiến lược chẵn/lẻ** thắng trước **mọi** cách đi của Bob (vét cạn cây Bob) cho `n ≤ 14`; và thống kê: nhóm chẵn lớn hơn hay nhóm lẻ lớn hơn đều xảy ra.

In [22]:
rng = random.Random(20240877)
even_wins = odd_wins = 0
for _ in range(400):
    n = rng.choice([2, 4, 6, 8, 10, 12, 14])
    while True:
        p = [rng.randint(1, 50) for _ in range(n)]
        if sum(p) % 2 == 1:
            break
    if n <= 12:
        check(p)
    assert parity_strategy_wins_all_bob_moves(p)
    assert stone_dp_memo(p) is True and stone_dp_table(p) is True
    if sum(p[0::2]) > sum(p[1::2]):
        even_wins += 1
    else:
        odd_wins += 1
print("400 ca ngẫu nhiên: chiến lược chẵn/lẻ thắng trước mọi cách đi của Bob; nhóm chẵn lớn hơn:", even_wins, "| nhóm lẻ lớn hơn:", odd_wins)
assert even_wins and odd_wins

400 ca ngẫu nhiên: chiến lược chẵn/lẻ thắng trước mọi cách đi của Bob; nhóm chẵn lớn hơn: 197 | nhóm lẻ lớn hơn: 203


### F. Giới hạn `n = 500`

DP bảng và có nhớ cho `True`; hiệu số hai bên cùng dấu; và công thức `Alice − Bob ≥ |E − O|` (Alice được ít nhất hiệu số của chiến lược chẵn/lẻ).

In [23]:
rng = random.Random(500)
for _ in range(20):
    p = random_valid(rng, 500)
    assert stone_dp_table(p) is True and Solution().stoneGame(p) is True
    n = len(p)
    dp = p[:]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i] = max(p[i] - dp[i + 1], p[j] - dp[i])
    margin = dp[0]
    assert margin >= abs(sum(p[0::2]) - sum(p[1::2]))                # chiến lược chẵn/lẻ cho Alice ít nhất |E - O|
print("20 đầu vào n = 500: Alice thắng; hiệu số tối ưu >= |E - O|")

20 đầu vào n = 500: Alice thắng; hiệu số tối ưu >= |E - O|


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tham lam (lấy đống lớn hơn ở hai đầu)

Trực giác "luôn lấy đống lớn hơn" sai trong trò chơi hai người.

In [24]:
bad = next(p for p in tests if bug_greedy_both(p) is not True)
print("tham lam cả hai:", bad, "->", bug_greedy_both(bad), "| minimax:", oracle(bad))
assert bug_greedy_both(bad) is False and oracle(bad)[0] > oracle(bad)[1]

tham lam cả hai: [4, 5, 7, 5] -> False | minimax: (11, 10)


**Vì sao sai:** lấy đống lớn ngay có thể mở ra đống lớn cho đối thủ. **Đúng:** minimax / DP, hoặc chiến lược chẵn/lẻ.

### Lỗi 2: Alice tham lam nhưng Bob tối ưu

Chỉ Alice tham lam cũng có thể thua.

In [25]:
bad = next(p for p in tests if bug_alice_greedy(p) is not True)
print("Alice tham lam:", bad, "->", bug_alice_greedy(bad), "| đúng True")
assert bug_alice_greedy(bad) is False

Alice tham lam: [4, 5, 7, 5] -> False | đúng True


**Vì sao sai:** Alice phải tính nước đáp trả của Bob. **Đúng:** `max(p[i] - f(i+1, j), p[j] - f(i, j-1))`.

### Lỗi 3: luôn chọn nhóm chẵn

Chiến lược chẵn/lẻ cần **chọn nhóm lớn hơn**.

In [26]:
bad = next(p for p in tests if bug_fixed_even(p) is not True)
print("luôn chẵn:", bad, "chẵn =", sum(bad[0::2]), "lẻ =", sum(bad[1::2]), "->", bug_fixed_even(bad))
assert bug_fixed_even(bad) is False

luôn chẵn: [8, 2, 1, 8] chẵn = 9 lẻ = 10 -> False


**Vì sao sai:** nhóm lẻ có thể lớn hơn; Alice có thể ép lấy nhóm lẻ. **Đúng:** so `E` và `O`, chọn lớn hơn (hoặc đơn giản trả `True`).

### Lỗi 4: cộng thay vì trừ giá trị của đối thủ

`f = max(p[i] + f(i+1, j), ...)` cộng **mọi** đống cho người đi hiện tại.

In [27]:
bad = next((p for p in tests if bug_add_not_sub(p) is not True), None)
print("cộng thay vì trừ, trên các đầu vào hợp lệ: luôn True?", bad is None)
print("với n lẻ [1, 5, 2]: bản cộng ->", bug_add_not_sub([1, 5, 2]), "(đúng là False vì hiệu số -2)")
assert bad is None and bug_add_not_sub([1, 5, 2]) is True

cộng thay vì trừ, trên các đầu vào hợp lệ: luôn True? True
với n lẻ [1, 5, 2]: bản cộng -> True (đúng là False vì hiệu số -2)


**Vì sao sai:** `f` trở thành tổng tất cả đống (luôn dương), nên luôn ra `True` kể cả khi Alice thật sự thua (ví dụ `[1, 5, 2]`, hiệu số `−2`). Trên đầu vào hợp lệ của bài nó trả `True` một cách "vô tình đúng", nhưng mô hình sai.

### Lỗi 5: kết luận `True` khi ràng buộc bị phá

Cách 4 chỉ đúng với `n` chẵn và tổng lẻ. Xem phản ví dụ ở mục 15.

### Lỗi 6 (bonus): cho phép lấy ở giữa

Luật chỉ cho lấy đầu hoặc cuối; mô hình hoá sai làm sai quy hoạch động.

In [28]:
print("lấy ở giữa tự do [3,7,2,3]:", bug_take_middle([3, 7, 2, 3]), "| trạng thái là tập con (2^n), không phải đoạn [i,j]")

lấy ở giữa tự do [3,7,2,3]: True | trạng thái là tập con (2^n), không phải đoạn [i,j]


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "hai người luân phiên, cả hai tối ưu, tổng cố định" → **minimax**, thường biến thành DP trên **hiệu số** (`f = điểm mình − điểm đối thủ`) với trạng thái là **đoạn** `[i, j]`. Nếu ràng buộc đặc biệt (chẵn/lẻ) thì thường có **lời giải toán** `O(1)`.

**Mẹo:**

1. Định nghĩa `f` theo góc nhìn **người đi hiện tại**; chuyển lượt thì đổi dấu (`−f`).
2. Trạng thái là đoạn liên tiếp ⇒ DP `O(n²)`.
3. Kiểm tra ràng buộc đặc biệt (chẵn/lẻ, tổng lẻ) để tìm lời giải đóng.
4. Tham lam thường **sai** trong trò chơi hai người.
5. Oracle: minimax vét cạn, mỗi người tối đa hoá số đá của mình.

**Họ bài cùng pattern:** Stone Game II (1140), III (1406), VII, Predict the Winner (486), Can I Win (464), Nim Game (292), Cat and Mouse.

## 15. Biến thể

1. **Hiệu số tối ưu** (trả `f(0, n−1)`), không chỉ thắng/thua.
2. **`n` lẻ** hoặc **tổng chẵn** (Predict the Winner 486): Alice thắng ⇔ `f ≥ 0`.
3. **Tìm đầu vào làm Alice thua** khi `n` lẻ.
4. **Thứ tự lấy tối ưu** của Alice (in nước đi).
5. **Chỉ được lấy tối đa 2 đống một lượt** (Stone Game II) — nhân rộng trạng thái.

Cài biến thể 1, 2 (`predict_the_winner`) và tìm phản ví dụ cho 3:

In [29]:
def optimal_margin(piles):
    n = len(piles)
    dp = piles[:]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i] = max(piles[i] - dp[i + 1], piles[j] - dp[i])
    return dp[0]


def predict_the_winner(nums):                             # LeetCode 486: Alice thắng hoặc hoà (f >= 0)
    return optimal_margin(nums) >= 0


# khớp oracle (hiệu số từ minimax) trên mọi đầu vào nhỏ, KỂ CẢ không hợp lệ (n lẻ, tổng chẵn)
cnt = 0
for n in range(1, 7):
    for p in itertools.product(range(1, 5), repeat=n):
        a, b = oracle(list(p))
        assert optimal_margin(list(p)) == a - b, p
        cnt += 1
print("hiệu số tối ưu khớp minimax trên", cnt, "đầu vào (mọi n <= 6, kể cả lẻ/chẵn)")

# 3) n lẻ: Alice có thể THUA
loser = next(list(p) for p in itertools.product(range(1, 6), repeat=3) if sum(p) % 2 == 1 and optimal_margin(list(p)) < 0)
print("n = 3 mà Alice thua:", loser, "-> hiệu số", optimal_margin(loser), "(Cách 4 sẽ sai ở đây)")
assert predict_the_winner([1, 5, 2]) is False and predict_the_winner([1, 5, 233, 7]) is True
# n chẵn, tổng chẵn: Alice không thua (f >= 0) nhưng có thể hoà
for n in (2, 4, 6):
    for p in itertools.product(range(1, 5), repeat=n):
        if sum(p) % 2 == 0:
            assert optimal_margin(list(p)) >= 0
print("n chẵn (tổng bất kỳ): hiệu số của Alice luôn >= 0; hoà xảy ra khi tổng chẵn, ví dụ", [1, 1], "->", optimal_margin([1, 1]))

hiệu số tối ưu khớp minimax trên 5460 đầu vào (mọi n <= 6, kể cả lẻ/chẵn)
n = 3 mà Alice thua: [1, 3, 1] -> hiệu số -1 (Cách 4 sẽ sai ở đây)


n chẵn (tổng bất kỳ): hiệu số của Alice luôn >= 0; hoà xảy ra khi tổng chẵn, ví dụ [1, 1] -> 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax vét cạn | thử mọi cây trò chơi | `O(2^n)` | `O(n)` | oracle `n` nhỏ |
| 2. DP có nhớ | `f(i,j)` theo hiệu số | `O(n²)` | `O(n²)` | tổng quát |
| 3. DP bảng 1 chiều | duyệt độ dài đoạn | `O(n²)` | `O(n)` | tổng quát, tiết kiệm |
| 4. `return True` | chiến lược chẵn/lẻ | `O(1)` | `O(1)` | **đáp án chính** (khi thoả ràng buộc) |

Cả bốn khớp minimax trên mọi đầu vào hợp lệ nhỏ (mục 12D); DP còn đúng cả khi ràng buộc bị phá (mục 15).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(optimal_margin([5, 3, 4, 5]), optimal_margin([1, 5, 2]), optimal_margin([3, 7, 2, 3]), predict_the_winner([1, 1]))

1 -2 5 True


### Bài 2 (tính tay)

`piles = [2, 9, 4, 6]`. (a) Tổng chỉ số chẵn và lẻ? (b) Alice nên ép nhóm nào? (c) Hiệu số tối ưu `f(0, 3)` bằng bao nhiêu? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP bảng:

In [31]:
def table_todo(piles):
    n = len(piles)
    dp = piles[:]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            pass        # TODO: dp[i] = max(piles[i] - dp[i+1], piles[j] - dp[i])
    return dp[0]


# Khi làm xong, bỏ comment để kiểm tra:
# assert table_todo([5, 3, 4, 5]) == 1

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP có nhớ** (`lru_cache`):

In [32]:
class MySolution:
    def stoneGame(self, piles):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().stoneGame(p) == w for p, w in examples)

### Bài 5 (mini challenge): in nước đi tối ưu của Alice

Viết `alice_first_move(piles)` trả `"L"` (lấy đầu) hoặc `"R"` (lấy cuối) là nước đầu tiên tối ưu của Alice (nếu hoà nước, ưu tiên `"L"`). Kiểm chứng: sau nước đó, hiệu số bằng `optimal_margin`.

In [33]:
def alice_first_move(piles):
    pass            # TODO


# Khi làm xong:
# assert alice_first_move([5, 3, 4, 5]) in ("L", "R")

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `f(i,j)` là hiệu số người đi trừ người kia.
- Mức 2: `[1,5,2]`: Alice lấy `2`, Bob `5`, Alice `1` → `3 − 5 = −2`; hoặc Alice lấy `1`, Bob `5`, Alice `2` → `−2`.
- Mức 3: kết quả `1 -2 5 True`: `[5,3,4,5]` → `1`, `[1,5,2]` → `−2`, `[3,7,2,3]` → `5`; `[1,1]` hoà (hiệu số `0`) nên `predict_the_winner = True`.

**Bài 2**

- Mức 1: chỉ số chẵn `0, 2`: `2 + 4 = 6`; lẻ `1, 3`: `9 + 6 = 15`.
- Mức 2: nhóm lẻ lớn hơn: Alice ép lấy chỉ số lẻ.
- Mức 3: hiệu số tối ưu ít nhất `15 − 6 = 9`; tính bằng DP để biết chính xác.

**Bài 3**

- Mức 1: `dp[i]` hiện là `f(i, j-1)`.
- Mức 2: `dp[i+1]` là `f(i+1, j)`.
- Mức 3: `dp[i] = max(piles[i] - dp[i+1], piles[j] - dp[i])`.

**Bài 4**

- Mức 1: `f(i, j)` có nhớ.
- Mức 2: cơ sở `i == j`.
- Mức 3: trả `f(0, n-1) > 0`.

**Bài 5**

- Mức 1: so hai nước: `piles[0] - f(1, n-1)` và `piles[n-1] - f(0, n-2)`.
- Mức 2: chọn nước lớn hơn (hoà thì `"L"`).
- Mức 3: dùng `optimal_margin` trên các đoạn con.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
assert optimal_margin([5, 3, 4, 5]) == 1 and optimal_margin([1, 5, 2]) == -2 and predict_the_winner([1, 1]) is True
assert optimal_margin([3, 7, 2, 3]) == oracle([3, 7, 2, 3])[0] - oracle([3, 7, 2, 3])[1]
print("Bài 1 OK:", optimal_margin([5, 3, 4, 5]), optimal_margin([1, 5, 2]), optimal_margin([3, 7, 2, 3]), True)

Bài 1 OK: 1 -2 5 True


### Bài 2

(a) chẵn `2 + 4 = 6`, lẻ `9 + 6 = 15`. (b) nhóm lẻ. (c) Hiệu số tối ưu tính bằng DP; nó ≥ `15 − 6 = 9`.

In [35]:
p = [2, 9, 4, 6]
assert (sum(p[0::2]), sum(p[1::2])) == (6, 15)
m = optimal_margin(p)
a, b = oracle(p)
assert m == a - b and m >= 15 - 6
print("Bài 2 OK: hiệu số tối ưu =", m, f"(Alice {a}, Bob {b})")

Bài 2 OK: hiệu số tối ưu = 9 (Alice 15, Bob 6)


### Bài 3

In [36]:
def table_answer(piles):
    n = len(piles)
    dp = piles[:]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i] = max(piles[i] - dp[i + 1], piles[j] - dp[i])
    return dp[0]


assert table_answer([5, 3, 4, 5]) == 1
rng = random.Random(3)
for _ in range(300):
    p = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]
    a, b = oracle(p)
    assert table_answer(p) == a - b
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def stoneGame(self, piles):
        @lru_cache(None)
        def f(i, j):
            if i == j:
                return piles[i]
            return max(piles[i] - f(i + 1, j), piles[j] - f(i, j - 1))

        return f(0, len(piles) - 1) > 0


assert all(MySolutionAnswer().stoneGame(p) == w for p, w in examples)
rng = random.Random(4)
for _ in range(300):
    n = rng.choice([2, 4, 6, 8])
    p = [rng.randint(1, 9) for _ in range(n)]
    a, b = oracle(p)
    assert MySolutionAnswer().stoneGame(p) == (a > b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def alice_first_move_answer(piles):
    n = len(piles)
    left = piles[0] - (optimal_margin(piles[1:]) if n > 1 else 0)
    right = piles[-1] - (optimal_margin(piles[:-1]) if n > 1 else 0)
    return "L" if left >= right else "R"


for p in ([5, 3, 4, 5], [3, 7, 2, 3], [2, 9, 4, 6]):
    mv = alice_first_move_answer(p)
    rest = p[1:] if mv == "L" else p[:-1]
    taken = p[0] if mv == "L" else p[-1]
    assert taken - optimal_margin(rest) == optimal_margin(p)
rng = random.Random(5)
for _ in range(300):
    p = [rng.randint(1, 9) for _ in range(rng.randint(2, 8))]
    mv = alice_first_move_answer(p)
    rest = p[1:] if mv == "L" else p[:-1]
    assert (p[0] if mv == "L" else p[-1]) - optimal_margin(rest) == optimal_margin(p)
print("Bài 5 OK: nước đi đầu đạt hiệu số tối ưu")

Bài 5 OK: nước đi đầu đạt hiệu số tối ưu


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (trò chơi hai người tổng cố định → DP hiệu số trên đoạn):**

```python
from functools import lru_cache

class Solution:
    def stoneGame(self, piles):
        @lru_cache(None)
        def f(i, j):                                   # (điểm người đi) - (điểm người kia) trên piles[i..j]
            if i == j:
                return piles[i]
            return max(piles[i] - f(i + 1, j), piles[j] - f(i, j - 1))
        return f(0, len(piles) - 1) > 0

        # Với ràng buộc của bài (n chẵn, tổng lẻ): return True là đủ.
```

**Checklist trước khi submit:**

- [ ] `f` định nghĩa theo góc nhìn **người đi hiện tại**, chuyển lượt thì **trừ** `f` (không cộng)?
- [ ] Cơ sở `f(i, i) = piles[i]`; trả `f(0, n−1) > 0`?
- [ ] Không dùng tham lam; không lấy đống ở giữa?
- [ ] Nếu trả `True`: chắc chắn `n` chẵn và tổng lẻ (ràng buộc của bài)?
- [ ] Đã thử `[5,3,4,5]`, `[3,7,2,3]`, `n = 2`, và (cho DP) một đầu vào `n` lẻ?
- [ ] Đúng tên `class Solution`, `stoneGame(self, piles)`, không in debug.